# 02 — Preprocessing & SOSA Mapping

**Goal:** Clean the 3W dataset and map it to the SOSA/SSN ontology for TKG ingestion.

## Steps:
1. Load and merge all instances
2. Handle NaN and frozen variables
3. Align timestamps (clock skew)
4. Map sensors to SOSA vocabulary
5. Add bitemporal annotation (valid_time + transaction_time)
6. Export preprocessed data for Neo4j loader

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

DATA_ROOT = DATA_ROOT_3W
OUTPUT_DIR = PROCESSED_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Colonne effettive nel dataset v2.0
SENSORS = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 
           'P-JUS-CKGL', 'T-JUS-CKGL', 'QGL', 'T-PDG', 'T-MON-CKP']

SOSA_MAPPING = {
    'P-PDG':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'downhole'},
    'P-TPT':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'subsea_christmas_tree'},
    'T-TPT':      {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'subsea_christmas_tree'},
    'P-MON-CKP':  {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'platform_upstream_pck'},
    'T-JUS-CKP':  {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'platform_downstream_pck'},
    'P-JUS-CKGL': {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'gas_lift_downstream'},
    'T-JUS-CKGL': {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'gas_lift_downstream'},
    'QGL':        {'observable_property': 'VolumetricFlowRate', 'unit': 'sm3_per_s', 'position': 'gas_lift_injection'},
}

SENSOR_ORDER = ['T-PDG', 'P-TPT', 'T-TPT', 'T-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'T-JUS-CKGL', 'QGL']

EVENT_TYPES = {0:'Normal',1:'Abrupt_BSW',2:'Spurious_DHSV',3:'Severe_Slugging',
               4:'Flow_Instability',5:'Rapid_Productivity_Loss',
               6:'Quick_Restriction_PCK',7:'Scaling_PCK',8:'Hydrate',9:'Undefined'}

print(f'✅ Config loaded')
print(f'   Data root: {DATA_ROOT}')
print(f'   Output dir: {OUTPUT_DIR}')

✅ Config loaded
   Data root: /home/obiaggi/3w_temp/dataset
   Output dir: /home/obiaggi/3w_processed


## 1. Load All Instances

In [2]:
def load_all_instances(data_root: Path):
    processed_dir = Path('/home/obiaggi/3w_processed')
    processed_files = sorted(processed_dir.glob('type_*.parquet'))
    
    if processed_files:
        print(f'✅ Using {len(processed_files)} pre-processed files')
        # Carica SOLO type_0 come campione per il preprocessing
        # Per la tesi non serve tutto insieme
        sample_types = [0]  # Normal + 3 anomalie rappresentative
        dfs = []
        for f in processed_files:
            type_num = int(f.stem.split('_')[1])
            if type_num in sample_types:
                df = pd.read_parquet(f)
                print(f'  Loaded {f.name}: {len(df):,} rows')
                dfs.append(df)
        combined = pd.concat(dfs).reset_index(drop=True)
        print(f'\n✅ Total: {len(combined):,} observations')
        return combined
    return None

df_raw = load_all_instances(DATA_ROOT)
if df_raw is not None:
    print(df_raw.head(3))

✅ Using 10 pre-processed files
  Loaded type_0.parquet: 12,158,183 rows

✅ Total: 12,158,183 observations
   ABER-CKGL  ABER-CKP  ESTADO-DHSV  ESTADO-M1  ESTADO-M2  ESTADO-PXO  \
0        NaN       NaN          1.0        1.0        0.0         0.0   
1        NaN       NaN          1.0        1.0        0.0         0.0   
2        NaN       NaN          1.0        1.0        0.0         0.0   

   ESTADO-SDV-GL  ESTADO-SDV-P  ESTADO-W1  ESTADO-W2  ...  QBS  QGL  \
0            0.0           1.0        1.0        0.0  ...  NaN  0.0   
1            0.0           1.0        1.0        0.0  ...  NaN  0.0   
2            0.0           1.0        1.0        0.0  ...  NaN  0.0   

   T-JUS-CKP  T-MON-CKP  T-PDG     T-TPT  class  state  event_type  \
0   84.64463        NaN    0.0  119.0781   <NA>   <NA>           0   
1   84.63828        NaN    0.0  119.0781   <NA>   <NA>           0   
2   84.63194        NaN    0.0  119.0781   <NA>   <NA>           0   

                 instance_id  
0  W

## 2. Handle NaN and Frozen Variables

In [3]:
def flag_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    df['quality_flag'] = 'ok'
    
    for sensor in SENSORS:
        if sensor not in df.columns:
            continue
        # Flag NaN
        nan_mask = df[sensor].isnull()
        df.loc[nan_mask, 'quality_flag'] = 'missing'
        
        # Flag frozen per instance
        frozen_instances = []
        for instance_id, group in df.groupby('instance_id', sort=False):
            vals = group[sensor].dropna()
            if len(vals) > 0 and vals.nunique() == 1:
                frozen_instances.append(instance_id)
        if frozen_instances:
            mask = df['instance_id'].isin(frozen_instances)
            df.loc[mask, 'quality_flag'] = 'frozen'
    
    print('Data quality flags:')
    print(df['quality_flag'].value_counts().to_string())
    return df

if df_raw is not None:
    df_flagged = flag_data_quality(df_raw)
    print('✅ Quality flagging completed')
else:
    print('⚠️  Load dataset first')

Data quality flags:
quality_flag
missing    12158183
✅ Quality flagging completed


## 3. Add Bitemporal Annotation

In [10]:
def add_bitemporality(df: pd.DataFrame) -> pd.DataFrame:
    tx_time = datetime.now(timezone.utc)
    df['valid_from'] = df['timestamp'] if 'timestamp' in df.columns else tx_time
    df['valid_to'] = None
    df['tx_time'] = tx_time
    print('✅ Bitemporal annotation added')
    return df

# Questa parte deve essere FUORI dalla funzione, allo stesso livello di "def"
if df_flagged is not None:
    df_bitemporal = add_bitemporality(df_flagged)
    print('✅ Bitemporal annotation completed')
else:
    print('⚠️  Run flag_data_quality first')

## 4. Export Preprocessed Data

In [11]:
def export_for_neo4j(df: pd.DataFrame, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save observations as Parquet
    output_path = output_dir / 'observations_3w.parquet'
    df.to_parquet(output_path, index=False)
    print(f'✅ Observations saved: {output_path}')
    print(f'   Shape: ({len(df):,} rows, {len(df.columns)} columns)')
    
    # Save SOSA mapping
    mapping_path = output_dir / 'sosa_mapping.json'
    with open(mapping_path, 'w') as f:
        json.dump(SOSA_MAPPING, f, indent=2)
    print(f'✅ SOSA mapping saved: {mapping_path}')
    
    # Save sensor order for PRECEDES relation
    order_path = output_dir / 'sensor_order.json'
    with open(order_path, 'w') as f:
        json.dump({'precedes_order': SENSOR_ORDER,
                   'justification': 'Physical order: downhole → subsea → surface (Vargas et al. 2019)'}, f, indent=2)
    print(f'✅ Sensor order saved: {order_path}')

if df_raw is not None:
    export_for_neo4j(df_bitemporal, OUTPUT_DIR)
    print('✅ Export completed - ready for Neo4j ingestion')
else:
    print('⚠️  Load dataset first')


NameError: name 'df_bitemporal' is not defined